<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/lab_action_classifier_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# CELL 1 — GOOGLE DRIVE + TRAINING PATH SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from google.colab import drive

import os


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive.mount(
    DRIVE_MOUNT,
    force_remount=False
)

assert os.path.exists(
    f"{DRIVE_MOUNT}/MyDrive"
), "Google Drive belum mounted"


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

SAVE_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/action_classifier"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. SUBDIRECTORIES
# =========================================================

RAW_DIR = (
    f"{SAVE_DIR}/raw"
)

PROCESSED_DIR = (
    f"{SAVE_DIR}/processed"
)

TEACHER_DIR = (
    f"{SAVE_DIR}/teacher"
)

TRUSTED_DIR = (
    f"{SAVE_DIR}/trusted"
)

REPAIR_DIR = (
    f"{SAVE_DIR}/repair"
)

BLIND_TEST_DIR = (
    f"{SAVE_DIR}/blind_test"
)

MODEL_DIR = (
    f"{SAVE_DIR}/models"
)

ONNX_DIR = (
    f"{MODEL_DIR}/onnx"
)

REPORT_DIR = (
    f"{SAVE_DIR}/reports"
)


for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    TEACHER_DIR,
    TRUSTED_DIR,
    REPAIR_DIR,
    BLIND_TEST_DIR,
    MODEL_DIR,
    ONNX_DIR,
    REPORT_DIR,
]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# =========================================================
# 4. DATASET PATHS
# =========================================================

MANUAL_SEED_PATH = (
    f"{RAW_DIR}/manual_seed_v1.jsonl"
)

PUBLIC_RAW_PATH = (
    f"{RAW_DIR}/public_seed_raw.jsonl"
)

NORMALIZED_DATASET_PATH = (
    f"{PROCESSED_DIR}/normalized_dataset_v1.jsonl"
)

DEDUP_DATASET_PATH = (
    f"{PROCESSED_DIR}/dedup_dataset_v1.jsonl"
)

TEACHER_LABELED_PATH = (
    f"{TEACHER_DIR}/teacher_labeled_v1.jsonl"
)

TEACHER_REJECTED_PATH = (
    f"{TEACHER_DIR}/teacher_rejected_v1.jsonl"
)

TRUSTED_TRAIN_PATH = (
    f"{TRUSTED_DIR}/action_classifier_trusted_v1.jsonl"
)

REPAIR_DATASET_PATH = (
    f"{REPAIR_DIR}/targeted_repair_v1.jsonl"
)

BLIND_TEST_PATH = (
    f"{BLIND_TEST_DIR}/blind_test_v1.jsonl"
)


# =========================================================
# 5. MODEL OUTPUT PATHS
# =========================================================

MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier.joblib"
)

BEST_MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier_best.joblib"
)

MLB_PATH = (
    f"{MODEL_DIR}/multilabel_binarizer.joblib"
)

ONNX_MODEL_PATH = (
    f"{ONNX_DIR}/action_classifier.onnx"
)


# =========================================================
# 6. REPORT PATHS
# =========================================================

METRICS_PATH = (
    f"{REPORT_DIR}/training_metrics.json"
)

PREDICTIONS_PATH = (
    f"{REPORT_DIR}/test_predictions.csv"
)

ERROR_ANALYSIS_PATH = (
    f"{REPORT_DIR}/error_analysis.csv"
)

LABEL_DISTRIBUTION_PATH = (
    f"{REPORT_DIR}/label_distribution.csv"
)

ONNX_PARITY_PATH = (
    f"{REPORT_DIR}/onnx_parity.json"
)

BENCHMARK_PATH = (
    f"{REPORT_DIR}/latency_benchmark.json"
)


# =========================================================
# 7. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 01 — ACTION CLASSIFIER STORAGE")
print("=" * 70)

print(
    "SAVE_DIR       :",
    SAVE_DIR
)

print(
    "Manual seed    :",
    MANUAL_SEED_PATH
)

print(
    "Trusted train  :",
    TRUSTED_TRAIN_PATH
)

print(
    "Repair dataset :",
    REPAIR_DATASET_PATH
)

print(
    "Model output   :",
    MODEL_PATH
)

print(
    "Best model     :",
    BEST_MODEL_PATH
)

print(
    "ONNX model     :",
    ONNX_MODEL_PATH
)

print(
    "Metrics        :",
    METRICS_PATH
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

!pip -q install \
    pandas \
    numpy \
    scikit-learn \
    datasets \
    huggingface_hub \
    tqdm \
    matplotlib \
    joblib \
    requests \
    skl2onnx \
    onnx \
    onnxruntime

print("Dependencies installed.")


# =========================================================
# IMPORTS
# =========================================================

import os
import re
import json
import time
import random
import hashlib
import warnings

import requests
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    MultiLabelBinarizer
)

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    hamming_loss,
)


# =========================================================
# RANDOM SEED
# =========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore"
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — ENVIRONMENT")
print("=" * 70)

print(
    "Random seed :",
    SEED
)

print(
    "NumPy       :",
    np.__version__
)

print(
    "Pandas      :",
    pd.__version__
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 3 — ACTION TAXONOMY + DATASET SCHEMA
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. ACTION LABELS
# =========================================================

LABELS = [
    "READ",
    "WRITE",
    "DELETE",
    "EXECUTE",
    "NETWORK",
    "INSTALL",
    "PRIVILEGED",
    "SYSTEM_CHANGE",
]


LABEL_TO_ID = {
    label: index
    for index, label in enumerate(LABELS)
}

ID_TO_LABEL = {
    index: label
    for label, index in LABEL_TO_ID.items()
}


# =========================================================
# 2. DATASET COLUMNS
# =========================================================

DATASET_COLUMNS = [
    "command",
    "description",
    "actions",
    "confidence",
    "ambiguous",
    "source",
    "split_origin",
]


# =========================================================
# 3. VALIDATION FUNCTION
# =========================================================

def validate_actions(actions):

    if not isinstance(
        actions,
        list
    ):
        return False

    if len(actions) == 0:
        return False

    if len(actions) != len(set(actions)):
        return False

    for action in actions:

        if action not in LABELS:
            return False

    return True


# =========================================================
# 4. SCHEMA
# =========================================================

SCHEMA = {

    "model": (
        "action_classifier"
    ),

    "version": (
        "v1"
    ),

    "task": (
        "multi_label_classification"
    ),

    "labels": (
        LABELS
    ),

    "columns": (
        DATASET_COLUMNS
    ),
}


SCHEMA_PATH = (
    f"{PROCESSED_DIR}/dataset_schema_v1.json"
)


with open(
    SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SCHEMA,
        f,
        indent=2,
        ensure_ascii=False
    )


# =========================================================
# 5. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TAXONOMY")
print("=" * 70)

for index, label in enumerate(LABELS):

    print(
        f"{index:2} -> {label}"
    )


print()
print(
    "Total labels :",
    len(LABELS)
)

print(
    "Task         :",
    "MULTI-LABEL"
)

print(
    "Schema       :",
    SCHEMA_PATH
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 4 — TEACHER API SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import requests
import time
import re
import json

from google.colab import userdata


# =========================================================
# API CONFIG
# =========================================================

BASE_URL = (
    "https://api.deepseek.com/chat/completions"
)

API_KEY = userdata.get(
    "DEEPSEEK"
)

MODEL = "deepseek-v4-flash"

TEMPERATURE = 0

TIMEOUT_CONNECT = 10
TIMEOUT_READ = 45


if not API_KEY:

    raise RuntimeError(
        "DEEPSEEK API key tidak ditemukan "
        "di Colab Secrets."
    )


# =========================================================
# TEACHER CLASSIFICATION PROMPT
# =========================================================

TEACHER_SYSTEM_PROMPT = """
You are a strict shell-command action classifier.

Your task is NOT to execute commands.

Your task is NOT to judge whether a command is malicious.

Your task is ONLY to identify what ACTIONS the command
would perform if executed.

The classification is MULTI-LABEL.

A command may have one or multiple action labels.

Allowed labels:

READ
WRITE
DELETE
EXECUTE
NETWORK
INSTALL
PRIVILEGED
SYSTEM_CHANGE


=========================================================
READ
=========================================================

The command reads, displays, lists, inspects, queries,
or retrieves local information without intentionally
modifying it.

Examples:

cat /etc/os-release
-> READ

ls -la /tmp
-> READ

systemctl status ssh
-> READ


=========================================================
WRITE
=========================================================

The command creates, copies, moves, overwrites,
appends, downloads, or otherwise writes data
to local storage.

Examples:

echo hello > output.txt
-> WRITE

cp source.txt backup.txt
-> READ + WRITE

curl <TEST_URL> -o file
-> NETWORK + WRITE


=========================================================
DELETE
=========================================================

The command removes files, directories, records,
or other persistent data.

Examples:

rm test.txt
-> DELETE

rm -r <TEMP_DIR>
-> DELETE


=========================================================
EXECUTE
=========================================================

The command launches, runs, evaluates, invokes,
or restarts executable code, programs, scripts,
commands, or services.

Examples:

python app.py
-> EXECUTE

bash script.sh
-> EXECUTE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
NETWORK
=========================================================

The command sends, receives, downloads, uploads,
queries, connects to, scans, or otherwise interacts
with a network or remote host.

Examples:

curl <TEST_URL>
-> NETWORK

ping <LAB_HOST>
-> NETWORK

wget <TEST_URL> -O file
-> NETWORK + WRITE


=========================================================
INSTALL
=========================================================

The command installs, adds, upgrades, or removes
software packages or software dependencies.

Examples:

pip install requests
-> NETWORK + INSTALL + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
PRIVILEGED
=========================================================

The command explicitly requests elevated privileges
or performs an operation requiring an elevated
privilege boundary.

Examples:

sudo apt update
-> NETWORK + PRIVILEGED + SYSTEM_CHANGE

sudo systemctl restart ssh
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
SYSTEM_CHANGE
=========================================================

The command modifies system state, configuration,
packages, services, permissions, users, system files,
mounts, firewall state, or other operating-system
configuration.

Examples:

chmod 600 file
-> SYSTEM_CHANGE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
IMPORTANT RULES
=========================================================

1. Return every action clearly implied by the command.

2. Do NOT classify based on whether the command is
   good, bad, suspicious, malicious, or dangerous.

3. Risk classification belongs to another model.

4. Focus only on observable command behavior.

5. Do not invent actions that are not implied.

6. If the command is genuinely unclear or cannot be
   reliably classified, set:

   "ambiguous": true

7. Confidence must reflect classification certainty.

8. Shell chaining must be classified across the
   entire command.

Example:

cat input.txt | curl -X POST <TEST_URL> -d @-

-> READ + NETWORK


=========================================================
OUTPUT
=========================================================

Return ONLY valid JSON.

Schema:

{
  "actions": ["LABEL"],
  "confidence": 0.95,
  "ambiguous": false
}

actions:
- must be a JSON array
- may contain one or multiple allowed labels
- must not contain duplicates

confidence:
- number from 0 to 1

ambiguous:
- true or false

Do not include explanations.
Do not include markdown.
Do not include reasoning.
"""


# =========================================================
# TEACHER CALL
# =========================================================

def call_teacher(command):

    headers = {

        "Authorization":
            f"Bearer {API_KEY}",

        "Content-Type":
            "application/json",
    }


    payload = {

        "model":
            MODEL,

        "temperature":
            TEMPERATURE,

        "thinking": {
            "type": "disabled"
        },

        "response_format": {
            "type": "json_object"
        },

        "messages": [

            {
                "role": "system",
                "content": TEACHER_SYSTEM_PROMPT,
            },

            {
                "role": "user",
                "content": command,
            },
        ],
    }


    start = time.time()


    response = requests.post(

        BASE_URL,

        headers=headers,

        json=payload,

        timeout=(
            TIMEOUT_CONNECT,
            TIMEOUT_READ
        ),
    )


    latency = (
        time.time()
        - start
    )


    response.raise_for_status()


    data = response.json()


    content = (
        data["choices"][0]
            ["message"]
            ["content"]
    )


    return content, latency


# =========================================================
# PARSER
# =========================================================

def parse_teacher_output(text):

    if not text:

        raise ValueError(
            "Teacher response kosong."
        )


    text = text.strip()


    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )


    text = re.sub(
        r"\s*```$",
        "",
        text
    )


    match = re.search(
        r"\{[\s\S]*?\}",
        text
    )


    if not match:

        raise ValueError(
            "JSON tidak ditemukan: "
            + text[:300]
        )


    data = json.loads(
        match.group()
    )


    actions = data.get(
        "actions",
        []
    )


    confidence = float(
        data.get(
            "confidence"
        )
    )


    ambiguous = data.get(
        "ambiguous"
    )


    # =====================================================
    # VALIDATE ACTIONS
    # =====================================================

    if not isinstance(
        actions,
        list
    ):

        raise ValueError(
            "actions harus berupa list."
        )


    actions = [

        str(action)
        .strip()
        .upper()

        for action in actions
    ]


    # Remove duplicates while preserving order

    actions = list(
        dict.fromkeys(actions)
    )


    if not actions:

        raise ValueError(
            "actions kosong."
        )


    for action in actions:

        if action not in LABELS:

            raise ValueError(
                f"Invalid action label: {action}"
            )


    # =====================================================
    # VALIDATE CONFIDENCE
    # =====================================================

    if not 0 <= confidence <= 1:

        raise ValueError(
            f"Invalid confidence: {confidence}"
        )


    # =====================================================
    # VALIDATE AMBIGUOUS
    # =====================================================

    if not isinstance(
        ambiguous,
        bool
    ):

        raise ValueError(
            "ambiguous harus boolean."
        )


    return {

        "actions":
            actions,

        "confidence":
            confidence,

        "ambiguous":
            ambiguous,
    }


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TEACHER API")
print("=" * 70)

print(
    "Model          :",
    MODEL
)

print(
    "API key        :",
    "OK"
)

print(
    "Labels         :",
    len(LABELS)
)

print(
    "call_teacher() :",
    "OK"
)

print(
    "parser         :",
    "OK"
)

print("=" * 70)